# Beginner tutorial: Habitat data, experiment configs, and the classical planner

This notebook follows **one real episode (114)** from the saved `hssd_fetch_stretch` experiment. It is deliberately slow and inspectable.

By the end, you should understand:

1. the difference between a Habitat/Hydra configuration and this experiment's `config.json`;
2. what information comes **into** the planner;
3. how robot skills become PDDL actions;
4. how Fast Downward chooses an action sequence and robot allocation;
5. what comes **out** of the planner and what happens during execution;
6. why a logically valid PDDL plan can still fail physically.

> This notebook does not start Habitat automatically. Most cells inspect saved artifacts, so they are safe and quick to run.

## 0. The system in one picture

```text
Habitat dataset episode
  ├─ scene, objects, target receptacles
  ├─ robot starting positions
  └─ ground-truth task goal
                 │
                 ▼
simulation feasibility trials (can each robot pick/place each object?)
                 │
                 ▼
PDDL domain + PDDL problem ──► Fast Downward ──► symbolic action plan
                                                     │
                                                     ▼
                                  MCP robot skills: navigate/pick/place
                                                     │
                                                     ▼
                                        Habitat execution + metrics
```

In `classical_only`, there is **no LLM**. The planner receives Habitat's ground-truth symbolic goal directly.

In [1]:
from pathlib import Path
import json
import sys
from pprint import pprint

# Find the repository whether Jupyter started in the repository root or examples-lab/.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT = next((p.resolve() for p in candidates if (p / 'mas').is_dir() and (p / 'results').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('Start Jupyter from habitat-my-mas or habitat-my-mas/examples-lab')

RUN_NAME = 'classical_only-fast-downward-opt__none__20260926-085415__r1'
RUN_DIR = PROJECT / 'results' / 'matrix' / 'hssd_fetch_stretch' / RUN_NAME
EPISODE_ID = '114'

print('Project:', PROJECT)
print('Run exists:', RUN_DIR.exists())
print('Episode:', EPISODE_ID)

Project: /home/rlwagun/files/simulation-test/habitat-my-mas
Run exists: True
Episode: 114


## 1. There are three different kinds of configuration

It is easy to call all of them “the config,” but they have different jobs.

| Configuration | Purpose | Typical contents |
|---|---|---|
| Habitat/Hydra YAML | Constructs the simulation and task | scene dataset, sensors, agents, actions, measures |
| Benchmark definition | Selects a reusable experiment world | Habitat YAML path, robot pool, dataset split |
| Saved run `config.json` | Records how one evaluation was launched | planner mode, engine, model, episode IDs, video |

The open `config.json` is therefore an **experiment receipt**. It does not itself construct the Habitat environment.

In [2]:
with (RUN_DIR / 'config.json').open() as f:
    run_config = json.load(f)

pprint(run_config)

{'benchmark': 'hssd_fetch_stretch',
 'config': 'fast-downward-opt',
 'engine': 'fast-downward-opt',
 'episodes': ['1173',
              '1003',
              '1067',
              '1140',
              '185',
              '371',
              '324',
              '114',
              '1158',
              '621',
              '665',
              '395',
              '1056',
              '1152',
              '615',
              '1139',
              '835',
              '471',
              '1042',
              '616',
              '1126',
              '133',
              '864',
              '411',
              '501',
              '182',
              '913',
              '269',
              '44',
              '140'],
 'feasibility_check': True,
 'fleet': ['fetch_0', 'stretch_0'],
 'instruction_level': 'explicit',
 'model': None,
 'normalize': False,
 'planner': 'classical_only',
 'prompt': None,
 'recovery': 'none',
 'repair': False,
 'repeat': 1,
 'run_id': 'classical_onl

In [3]:
# Translate the most important fields into plain English.
meaning = {
    'benchmark': 'Which Habitat task/fleet setup was used',
    'planner': 'Which high-level planning pipeline was used',
    'engine': 'The search engine solving the PDDL problem',
    'model': 'Language model; None means no LLM was called',
    'fleet': 'Robots available in every episode',
    'episodes': 'Dataset episode IDs evaluated by this run',
    'feasibility_check': 'Whether physical pick/place trials ran before planning',
    'recovery': 'What happens after an execution failure',
    'video': 'Whether execution video was recorded',
}
for key, description in meaning.items():
    value = run_config.get(key)
    if key == 'episodes':
        value = f'{len(value)} episodes; first five = {value[:5]}'
    print(f'{key:20} = {str(value):55} # {description}')

benchmark            = hssd_fetch_stretch                                      # Which Habitat task/fleet setup was used
planner              = classical_only                                          # Which high-level planning pipeline was used
engine               = fast-downward-opt                                       # The search engine solving the PDDL problem
model                = None                                                    # Language model; None means no LLM was called
fleet                = ['fetch_0', 'stretch_0']                                # Robots available in every episode
episodes             = 30 episodes; first five = ['1173', '1003', '1067', '1140', '185'] # Dataset episode IDs evaluated by this run
feasibility_check    = True                                                    # Whether physical pick/place trials ran before planning
recovery             = none                                                    # What happens after an execution failure

### Important conclusion

```text
planner = classical_only
model   = None
```

means the natural-language instruction is shown for human readability, but it is **not interpreted by a language model**. The code uses the episode's existing PDDL goal.

## 2. Understanding the saved dataset episode

`records.jsonl` means **JSON Lines**: every line is a complete JSON object representing one evaluated episode. Unlike a normal JSON file, the entire file is not surrounded by `[` and `]`.

The record combines four stages:

- input: instruction, goal and initial facts;
- planning: feasibility and the generated plan;
- execution: trace of actual robot skill calls;
- output: final facts and metrics.

In [4]:
records = []
with (RUN_DIR / 'records.jsonl').open() as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

episode = next(r for r in records if str(r['episode_id']) == EPISODE_ID)
print(f'Loaded {len(records)} episode records')
print('Fields in one record:')
print(sorted(episode.keys()))

Loaded 30 episode records
Fields in one record:
['benchmark', 'config', 'episode_id', 'execution_wall_s', 'feasibility', 'final_facts', 'fleet', 'goal', 'initial_facts', 'instruction', 'metrics', 'model', 'pipeline', 'plan', 'plan_errors', 'planner', 'repeat', 'run_id', 'trace', 'video']


In [7]:
print('HUMAN-READABLE INSTRUCTION')
print(episode['instruction'])

print('\nFORMAL GOAL FACTS')
for fact in episode['goal']:
    print(' ', fact)

print('\nINITIAL SYMBOLIC STATE')
for fact in episode['initial_facts']:
    print(' ', fact)

HUMAN-READABLE INSTRUCTION
Tidy up: put potted_meat_can_0 on table_0; put potted_meat_can_1 on couch_0. When done, no robot should be holding anything.

FORMAL GOAL FACTS
  (obj-at potted_meat_can_0 table_0)
  (obj-at potted_meat_can_1 couch_0)
  (hand-empty fetch_0)
  (hand-empty stretch_0)

INITIAL SYMBOLIC STATE
  (robot-at fetch_0 start_fetch_0)
  (hand-empty fetch_0)
  (robot-at stretch_0 start_stretch_0)
  (hand-empty stretch_0)
  (obj-at potted_meat_can_0 bed_0)
  (obj-at potted_meat_can_1 chair_0)


### How to read a fact

A PDDL fact is a small statement about the world:

```lisp
(obj-at potted_meat_can_0 table_0)
```

- `obj-at` is the predicate (the relationship);
- `potted_meat_can_0` is the object;
- `table_0` is the desired location.

The planner's job is to find actions whose effects transform the initial facts until every goal fact is true.

## 3. Feasibility connects symbolic planning to robot geometry

PDDL alone does not know whether an arm can reach a couch or table. Before planning, the simulator tries `navigate → pick → navigate → place` for every robot/object pair and records result codes.

In [6]:
feasibility = episode['feasibility']
for robot, objects in feasibility.items():
    print(f'\n{robot}')
    for obj, result in objects.items():
        print(f"  {obj:24} pick={result['pick']:15} place={result['place']}")


fetch_0
  potted_meat_can_0        pick=OK              place=OK
  potted_meat_can_1        pick=OK              place=OK

stretch_0
  potted_meat_can_0        pick=OK              place=OUT_OF_REACH
  potted_meat_can_1        pick=OK              place=OUT_OF_REACH


For episode 114, Stretch can pick both cans but cannot place them at their required destinations. A robot is considered capable of an object's complete task only when **both** pick and place are `OK`. Therefore Fast Downward receives only Fetch as a valid complete assignment.

## 4. Robot skills become a PDDL domain

The domain describes general rules, independent of episode 114. For example:

```text
pick is allowed when:
  robot and object are at the same symbolic location
  the robot's hand is empty
  feasibility says the robot can pick the object
  the object was assigned to that robot

pick changes the state to:
  robot is holding the object
  object is no longer at the old location
  robot's hand is no longer empty
```

In [ ]:
# Import the project's lightweight skill catalog (this does not start Habitat).
sys.path.insert(0, str(PROJECT))
from mas.skills.catalog import SKILLS

for name in ['navigate_to', 'pick', 'place']:
    skill = SKILLS[name]
    print(f'\n=== {name} ===')
    print('Required low-level capabilities:', skill['requires'])
    pprint(skill['pddl'])

## 5. Inspect the exact PDDL sent to Fast Downward

`plan.json` preserves both generated PDDL strings. This is extremely useful for debugging: you can see exactly what the symbolic solver knew, rather than what we intended it to know.

In [ ]:
with (RUN_DIR / 'episodes' / EPISODE_ID / 'plan.json').open() as f:
    saved_plan = json.load(f)

planner_call = saved_plan['planner_calls'][0]
print('Engine:', planner_call['engine'])
print('Planning latency:', planner_call['latency_s'], 'seconds')
print('LLM tokens:', planner_call['prompt_tokens'], '+', planner_call['completion_tokens'])
print('Infeasible objects:', planner_call['infeasible_objects'])

In [ ]:
print('----- GENERATED DOMAIN -----')
print(planner_call['domain'])
print('\n----- GENERATED EPISODE PROBLEM -----')
print(planner_call['problem'])

When reading the problem, find these four sections:

1. `:objects` — names and types known to the planner;
2. `:init` — facts true before execution, including feasibility and costs;
3. `:goal` — facts that must be true at the end;
4. `:metric` — tells the solver to minimize total action cost.

## 6. Allocation and optimization

The code enumerates feasible object-to-robot assignments (up to 64). It solves a PDDL problem for each assignment, estimates the workload per robot, then minimizes:

```text
1. makespan = the largest workload assigned to any one robot
2. total    = total estimated work across all robots (tie-breaker)
```

This encourages two robots to work concurrently when both are feasible.

In [ ]:
for number, trial in enumerate(planner_call['allocations'], start=1):
    print(f'Allocation {number}')
    pprint(trial)

`SOLVED_OPTIMALLY` means optimal for the **simplified symbolic cost model**. It does not guarantee that Habitat navigation or manipulation will succeed. The navigation cost is an estimate based on straight-line horizontal distance; physical actions can still encounter obstacles, imperfect arrival poses, or IK failures.

## 7. Optional: solve the saved PDDL again

This cell invokes Unified Planning and Fast Downward but still does **not** launch Habitat. If the notebook kernel does not use the project's `.venv`, the cell will print installation guidance instead of stopping the tutorial.

In [ ]:
try:
    from unified_planning.io import PDDLReader
    from unified_planning.shortcuts import OneshotPlanner, get_environment

    get_environment().credits_stream = None
    problem = PDDLReader().parse_problem_string(
        planner_call['domain'], planner_call['problem']
    )
    with OneshotPlanner(name=run_config['engine']) as planner:
        result = planner.solve(problem)
    print('Solver status:', result.status)
    print('\nActions returned by the solver:')
    for action in result.plan.actions:
        print(' ', action)
except ImportError as exc:
    print('Planner packages are not installed in this notebook kernel.')
    print('Launch Jupyter with habitat-my-mas/.venv or install requirements.txt.')
    print('Details:', exc)

## 8. Planner output: raw PDDL actions versus robot skills

Fast Downward returns actions with symbolic bookkeeping parameters. The project converts them into the smaller MCP calls accepted by each robot.

In [ ]:
print('RAW FAST DOWNWARD ACTIONS')
for action in saved_plan['raw']:
    print(' ', action)

print('\nCONVERTED ROBOT SKILL QUEUES')
for robot, steps in saved_plan['steps'].items():
    print(f'\n{robot}:')
    for index, step in enumerate(steps):
        print(f"  {index}: {step['skill']}({step['args']})")

Execution follows two timing rules:

- steps belonging to one robot run sequentially;
- different robots run concurrently.

A robot stops its queue after its first failed skill. In this saved run, recovery was `none`, so the system does not replan after failure.

## 9. Execution trace: what actually happened

The plan is an intention. The trace is the actual sequence of robot tool results returned during simulation. Always inspect both.

In [ ]:
for event in episode['trace']:
    marker = '✓' if event['status'] == 'ok' else '✗'
    print(
        f"{marker} t={event['t_start']:>5.2f}–{event['t_end']:>5.2f}s  "
        f"{event['robot']:10}  {event['skill']:11}  "
        f"{event['args']}  -> {event['code']} ({event['message']})"
    )

In [ ]:
print('FINAL SYMBOLIC FACTS')
for fact in episode['final_facts']:
    print(' ', fact)

print('\nGOAL CHECKS')
for goal, met in episode['metrics']['goal_met'].items():
    print(' ✓' if met else ' ✗', goal)

print('\nSUMMARY METRICS')
for key in ['success', 'success_symbolic', 'subgoal_rate', 'plan_valid',
            'plan_steps', 'steps_ok', 'steps_failed', 'failure_codes',
            'planning_latency_s', 'prompt_tokens', 'completion_tokens']:
    print(f"{key:22} {episode['metrics'][key]}")

### Why episode 114 failed

The final `place` call returned `OUT_OF_REACH`. Geometrically, the can was close enough to the couch for Habitat's object-location subgoal to be true, but Fetch was still considered to be holding it. Thus:

```text
object on couch goal       = true
Fetch hand-empty goal      = false
overall success            = false
subgoal rate               = 3 / 4 = 0.75
```

This is the central planning lesson: **symbolic validity is not physical reliability**. The PDDL model predicts logical state changes; the simulator executes continuous motion and can fail.

## 10. Dataset input and output summary

| Stage | Input | Output |
|---|---|---|
| Habitat reset | episode ID + active robots | objects, poses, goal receptacles, robot state |
| Feasibility | robot/object pairs | `pick` and `place` result codes |
| Problem builder | state + goal + feasibility | PDDL domain and problem |
| Fast Downward | PDDL domain and problem | ordered symbolic actions |
| Plan converter | symbolic actions | per-robot MCP skill queues |
| Executor | skill queues | trace of status/code/message/steps |
| Evaluator | final Habitat state + goal | success and diagnostic metrics |

The original compressed Habitat dataset contains episode specifications. The files under `results/` are **evaluation outputs derived from those episodes**, not the source dataset itself.

## 11. Optional live run

Only do this after the offline cells make sense. Use two terminals from the repository root.

**Terminal 1 — Habitat environment:**

```bash
conda activate habitat-mas
python -m mas.sim_host.server --benchmark hssd_fetch_stretch
```

**Terminal 2 — planner environment:**

```bash
.venv/bin/python -m mas.eval.run \
  --planner classical_only \
  --engine fast-downward-opt \
  --episode-ids 114
```

A repeated physical run may not reproduce every manipulation outcome because navigation and arm endpoints can differ slightly. Compare the new `plan.json`, `trace`, and video rather than looking only at `success`.

## 12. Beginner exercises

Do these in order; they require no source-code changes.

1. Change `EPISODE_ID` to `1140`. Which robot receives each object, and why?
2. Pick an episode where both robots have `pick=OK` and `place=OK`. How many allocations were tried?
3. In the PDDL problem, locate every `assigned`, `can-pick`, and `can-place` fact. Explain which component generated each one.
4. Compare estimated `makespan` with actual `sim_steps`. Why are they different?
5. Find a record where `plan_valid=True` and `success=False`. Use its trace to identify the symbolic/physical gap.
6. Compare this classical-only run with an `slm_classical` run. Identify the extra LLM call and verify that Fast Downward still performs allocation and ordering.

### Source files to read next

1. `mas/skills/catalog.py` — skill meaning, preconditions and effects;
2. `mas/planners/classical.py` — problem construction, allocation and solving;
3. `mas/eval/run.py` — full episode pipeline and metrics;
4. `mas/sim_host/host.py` — simulator state and feasibility trials.

Read one function at a time while keeping episode 114's `plan.json` open beside it.